# compare_sources — Western vs Islamic Republic framing

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel. See `README.md` → Environment setup.

**Input (read-only):** English `data/output/edges/` + `data/interim/corpus_clean.jsonl`, and Islamic Republic `data/output/iran/edges/` + `data/interim/iran/corpus_clean.jsonl`
**Output:** `data/output/compare/` (CSVs + figures)

Puts the two independently-built co-occurrence networks side by side for the **same** actors, concepts, and time windows. It **reads** both corpora and writes only to `data/output/compare/` — neither corpus's files are modified.

**Volume asymmetry:** the English corpus is much larger, so all comparisons use `weight_normalized` (co-occurrences **per article in that corpus's window**), and a corpus-stats table makes the raw volume gap explicit. A denser Western network is not "more framing" — it is more text.

**Trust caveat (A5):** the Iranian alias dictionary (`src/alias_map_iran`) is a domain-knowledge starting point; until the 30–50-sentence audit of IRNA/Tasnim is done and the alias map extended, treat the Iranian side as provisional. (`04_extract_iran` prints the top alias misses to drive that audit.)

## Steps
1. Setup & paths
2. Load both edge sets + per-window article counts
3. Corpus stats (volume asymmetry)
4. Edge-level framing comparison (biggest divergences)
5. Key-dyad comparison over time
6. Actor concept-profile comparison
7. Validation checkpoint

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_cwd = Path().resolve()
ROOT = next((p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()), _cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.concept_dict import CONCEPT_DICT
from src.time_windows import TIME_WINDOWS

WINDOW_ORDER = [w[0] for w in TIME_WINDOWS]
CONCEPTS = list(CONCEPT_DICT.keys())

EN = {'label': 'Western', 'edges': ROOT / 'data' / 'output' / 'edges',
      'clean': ROOT / 'data' / 'interim' / 'corpus_clean.jsonl'}
IR = {'label': 'Islamic Republic', 'edges': ROOT / 'data' / 'output' / 'iran' / 'edges',
      'clean': ROOT / 'data' / 'interim' / 'iran' / 'corpus_clean.jsonl'}

EN['analysis'] = ROOT / 'data' / 'output' / 'analysis'
IR['analysis'] = ROOT / 'data' / 'output' / 'iran' / 'analysis'

COMPARE_DIR = ROOT / 'data' / 'output' / 'compare'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

print(f'English edges : {EN["edges"]}')
print(f'Iranian edges : {IR["edges"]}')
print(f'Comparison out: {COMPARE_DIR}')
assert EN['clean'].exists(), "English corpus missing — run ./run_all.command first"

## Step 2: Load both edge sets + article counts

Both corpora share the analytic frame (`ACTOR_WHITELIST`, `CONCEPT_DICT`, `TIME_WINDOWS`) so the merge is on identical `(window, actor, concept)` keys. `weight_normalized` is per-article within each corpus, so it is directly comparable across corpora.

In [ ]:
def load_edges(d):
    rows = []
    for f in sorted(Path(d).glob('edges_*.jsonl')):
        with open(f, encoding='utf-8') as fh:
            rows += [json.loads(line) for line in fh]
    return rows

def article_counts(path):
    c = Counter()
    if Path(path).exists():
        with open(path, encoding='utf-8') as fh:
            for line in fh:
                c[json.loads(line)['window']] += 1
    return c

en_edges = load_edges(EN['edges'])
ir_edges = load_edges(IR['edges'])
en_arts = article_counts(EN['clean'])
ir_arts = article_counts(IR['clean'])

IR_PRESENT = len(ir_edges) > 0
print(f'English edges loaded : {len(en_edges)}  ({sum(en_arts.values())} articles)')
print(f'Iranian edges loaded : {len(ir_edges)}  ({sum(ir_arts.values())} articles)')
if not IR_PRESENT:
    print('\n*** Iranian corpus not yet built — run ./run_all_iran.command, then re-run this notebook. ***')
    print('    (Steps below render only the English side / are skipped.)')

## Step 3: Corpus stats — the volume asymmetry, made explicit

In [ ]:
stats = pd.DataFrame({
    'western_articles': pd.Series(en_arts),
    'iran_articles':    pd.Series(ir_arts),
}).reindex(WINDOW_ORDER).fillna(0).astype(int)
stats.loc['TOTAL'] = stats.sum()
stats['iran_share'] = (stats['iran_articles'] /
                       (stats['western_articles'] + stats['iran_articles']).replace(0, np.nan)).round(3)
stats.to_csv(COMPARE_DIR / 'compare_corpus_stats.csv')
print('Saved: compare_corpus_stats.csv')
print(stats.to_string())
print(f'\nOverall volume ratio (Western : Iranian) = '
      f'{sum(en_arts.values()) / max(sum(ir_arts.values()), 1):.1f} : 1  '
      '-> all comparisons below use per-article normalized weights.')

## Step 4: Edge-level framing comparison

Merge the two edge sets on `(window, actor, concept)` and compare per-article weights. `wn_diff = wn_iran − wn_western` (positive = the Iranian corpus frames that actor–concept link *more strongly per article* than the Western corpus).

In [ ]:
if IR_PRESENT:
    en_df = (pd.DataFrame(en_edges)[['window', 'actor', 'concept', 'weight_normalized']]
             .rename(columns={'weight_normalized': 'wn_western'}))
    ir_df = (pd.DataFrame(ir_edges)[['window', 'actor', 'concept', 'weight_normalized']]
             .rename(columns={'weight_normalized': 'wn_iran'}))
    merged = pd.merge(en_df, ir_df, on=['window', 'actor', 'concept'], how='outer').fillna(0.0)
    merged['wn_diff'] = (merged['wn_iran'] - merged['wn_western']).round(4)
    merged = merged.sort_values('wn_diff')
    merged.to_csv(COMPARE_DIR / 'compare_edges.csv', index=False)
    print(f'Saved: compare_edges.csv  ({len(merged)} actor-concept-window rows)')

    agg = (merged.groupby(['actor', 'concept'])[['wn_western', 'wn_iran']].sum())
    agg['wn_diff'] = (agg['wn_iran'] - agg['wn_western']).round(3)
    agg = agg.sort_values('wn_diff')
    print('\n=== Framed MORE by the Iranian corpus (per article, summed over windows) ===')
    print(agg.tail(10)[::-1].round(3).to_string())
    print('\n=== Framed MORE by the Western corpus ===')
    print(agg.head(10).round(3).to_string())
else:
    print('Skipped — Iranian corpus not present.')

## Step 5: Key-dyad comparison over time

The five thesis dyads, Western (solid) vs Islamic Republic (dashed), in per-article normalized weight across windows.

In [ ]:
KEY_DYADS = [('USA', 'military_action'), ('IRAN', 'deterrence'),
             ('IAEA', 'nuclear_program'), ('USA', 'strike_claims'), ('IRAN', 'diplomacy')]

def series(edges, actor, concept):
    m = {w: 0.0 for w in WINDOW_ORDER}
    for e in edges:
        if e['actor'] == actor and e['concept'] == concept:
            m[e['window']] = e['weight_normalized']
    return [m[w] for w in WINDOW_ORDER]

if IR_PRESENT:
    x = np.arange(len(WINDOW_ORDER))
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, (a, c) in zip(axes.flat, KEY_DYADS):
        ax.plot(x, series(en_edges, a, c), marker='o', color='#1f8fa6', linewidth=2, label='Western')
        ax.plot(x, series(ir_edges, a, c), marker='s', color='#d64545', linewidth=2,
                linestyle='--', label='Islamic Republic')
        for i, w in enumerate(WINDOW_ORDER):
            if w.startswith('climax'):
                ax.axvspan(i - 0.5, i + 0.5, color='#7e57c2', alpha=0.10, zorder=0)
        ax.set_title(f'{a} — {c}', fontsize=10)
        ax.set_xticks(x); ax.set_xticklabels(WINDOW_ORDER, rotation=40, ha='right', fontsize=7)
        ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)
    axes.flat[-1].axis('off')
    fig.suptitle('Key actor–concept dyads: Western vs Islamic Republic (per-article weight)', fontsize=13)
    plt.tight_layout()
    out = COMPARE_DIR / 'compare_key_dyads.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {out.name}')
else:
    print('Skipped — Iranian corpus not present.')

## Step 6: Actor concept-profile comparison

For key actors, the **share** of each concept in that actor's total per-article weight — does the Iranian corpus foreground different framings (e.g. strike_claims / deterrence) for the same actor?

In [ ]:
PROFILE_ACTORS = ['IRAN', 'USA', 'ISRAEL']

def profile(edges, actor):
    tot = Counter()
    for e in edges:
        if e['actor'] == actor:
            tot[e['concept']] += e['weight_normalized']
    s = sum(tot.values()) or 1.0
    return [tot.get(c, 0.0) / s for c in CONCEPTS]

if IR_PRESENT:
    fig, axes = plt.subplots(1, len(PROFILE_ACTORS), figsize=(16, 5), sharey=True)
    y = np.arange(len(CONCEPTS)); h = 0.38
    for ax, actor in zip(axes, PROFILE_ACTORS):
        ax.barh(y - h/2, profile(en_edges, actor), h, color='#1f8fa6', label='Western')
        ax.barh(y + h/2, profile(ir_edges, actor), h, color='#d64545', label='Islamic Republic')
        ax.set_yticks(y); ax.set_yticklabels(CONCEPTS)
        ax.set_title(actor); ax.set_xlabel('share of actor\'s concept weight')
        ax.grid(axis='x', alpha=0.3); ax.legend(fontsize=8)
    fig.suptitle('Concept profile per actor: Western vs Islamic Republic', fontsize=13)
    plt.tight_layout()
    out = COMPARE_DIR / 'compare_actor_profiles.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {out.name}')
else:
    print('Skipped — Iranian corpus not present.')

## Step 7: Multi-level comparison (co-occurrence)

Both corpora now run the same notebook 07, so the whole multi-level stack is
comparable — not just the dyad level. Three levels are put side by side here:

- **Framing polarity (volume share)** — the share of *mentions* that invoke
  conflictual vs. cooperative concepts. This is the sharpest cross-corpus
  contrast available.
- **Coalition share of voice** — what fraction of each corpus's framing volume
  belongs to us-aligned vs. iran-aligned actors. Share-based, so the 4× corpus
  size difference cancels out.
- **Graph-level saturation** — bipartite density and Latapy clustering: how
  broadly each outlet class spreads actors across framing dimensions.

**Deliberately excluded: the SVO layer.** The Iranian SVO network rests on 40
actor→actor triples across the whole corpus (2–7 edges per window, some windows
empty), so directed metrics — agent/target ratios, reciprocity, triads — are not
comparable between corpora and are not reported here.

In [ ]:
def load_level(side, rel):
    p = side['analysis'] / rel
    return pd.read_csv(p) if p.exists() else None

rows_ok = IR_PRESENT and (IR['analysis'] / 'polarity' / 'polarity_over_time.csv').exists()
if not rows_ok:
    print('Iranian analysis outputs missing — run ./run_all_iran.command (07) first.')
else:
    # ---------- 7a. polarity volume share ----------
    pol = {}
    for side in (EN, IR):
        d = load_level(side, 'polarity/polarity_over_time.csv').set_index('window')
        pol[side['label']] = d
    windows = [w for w in WINDOW_ORDER if w in pol['Western'].index and w in pol['Islamic Republic'].index]

    pol_cmp = pd.DataFrame({
        'western_negative': pol['Western'].loc[windows, 'volume_negative'],
        'iranian_negative': pol['Islamic Republic'].loc[windows, 'volume_negative'],
        'western_positive': pol['Western'].loc[windows, 'volume_positive'],
        'iranian_positive': pol['Islamic Republic'].loc[windows, 'volume_positive'],
    })
    pol_cmp['negative_gap'] = pol_cmp.iranian_negative - pol_cmp.western_negative
    pol_cmp.to_csv(COMPARE_DIR / 'compare_polarity.csv')
    print('=== Framing polarity, volume share (conflictual concepts) ===')
    print(pol_cmp.round(3).to_string())

    # ---------- 7b. coalition share of voice ----------
    grp = {}
    for side in (EN, IR):
        d = load_level(side, 'group_level/group_volume_share.csv')
        grp[side['label']] = d.pivot(index='window', columns='group', values='volume_share')
    grp_cmp = pd.DataFrame({
        'western_us': grp['Western'].loc[windows, 'us_aligned'],
        'iranian_us': grp['Islamic Republic'].loc[windows, 'us_aligned'],
        'western_iran': grp['Western'].loc[windows, 'iran_aligned'],
        'iranian_iran': grp['Islamic Republic'].loc[windows, 'iran_aligned'],
    })
    grp_cmp.to_csv(COMPARE_DIR / 'compare_group_share.csv')
    print('\n=== Coalition share of framing volume ===')
    print(grp_cmp.round(3).to_string())

    # ---------- 7c. graph-level saturation ----------
    gl = {}
    for side in (EN, IR):
        d = load_level(side, 'graph_level/graph_level_metrics.csv').set_index('window')
        gl[side['label']] = d
    gl_cmp = pd.DataFrame({
        'western_density': gl['Western'].loc[windows, 'density'],
        'iranian_density': gl['Islamic Republic'].loc[windows, 'density'],
        'western_clustering': gl['Western'].loc[windows, 'bipartite_clustering'],
        'iranian_clustering': gl['Islamic Republic'].loc[windows, 'bipartite_clustering'],
    })
    gl_cmp.to_csv(COMPARE_DIR / 'compare_graph_level.csv')
    print('\n=== Graph-level saturation ===')
    print(gl_cmp.round(3).to_string())

    # ---------- figure ----------
    x = np.arange(len(windows))
    W_C, I_C = '#1f8fa6', '#b3541e'      # teal = Western, sienna = Iranian
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    ax = axes[0]
    ax.plot(x, pol_cmp.western_negative, marker='o', lw=2.4, color=W_C, label='Western')
    ax.plot(x, pol_cmp.iranian_negative, marker='s', lw=2.4, color=I_C, label='Islamic Republic')
    ax.set_title('Conflictual framing (share of mentions)', fontsize=10)
    ax.set_ylabel('share'); ax.legend(fontsize=8)

    # Middle panel carries FOUR series in one axes, so colour alone cannot
    # separate them. Encoding: colour = corpus (as in the other two panels),
    # filled marker + solid line = us-aligned, hollow marker + dashed line =
    # iran-aligned. Every pair therefore differs in two channels at once.
    ax = axes[1]
    ax.plot(x, grp_cmp.western_us, marker='o', ms=6, lw=2.2, color=W_C,
            label='us-aligned · Western')
    ax.plot(x, grp_cmp.iranian_us, marker='s', ms=6, lw=2.2, color=I_C,
            label='us-aligned · Iranian')
    ax.plot(x, grp_cmp.western_iran, marker='o', ms=6, lw=1.8, ls=(0, (5, 2)),
            color=W_C, mfc='white', mew=1.6, label='iran-aligned · Western')
    ax.plot(x, grp_cmp.iranian_iran, marker='s', ms=6, lw=1.8, ls=(0, (5, 2)),
            color=I_C, mfc='white', mew=1.6, label='iran-aligned · Iranian')
    ax.set_title('Coalition share of voice', fontsize=10)
    lo = min(grp_cmp.min()); hi = max(grp_cmp.max())
    ax.set_ylim(lo - .03, hi + .16)          # headroom so the legend clears the lines
    ax.legend(fontsize=7.5, ncol=2, loc='upper center', handlelength=3.2,
              columnspacing=1.0, framealpha=.92)

    ax = axes[2]
    ax.plot(x, gl_cmp.western_density, marker='o', lw=2.4, color=W_C, label='Western')
    ax.plot(x, gl_cmp.iranian_density, marker='s', lw=2.4, color=I_C, label='Islamic Republic')
    ax.set_title('Bipartite density (framing breadth)', fontsize=10); ax.legend(fontsize=8)

    for ax in axes:
        for i, w in enumerate(windows):
            if w.startswith('climax'):
                ax.axvspan(i - .5, i + .5, color='#7e57c2', alpha=.10, zorder=0)
        ax.set_xticks(x); ax.set_xticklabels(windows, rotation=45, ha='right', fontsize=7)
        ax.grid(axis='y', alpha=.3)
    fig.suptitle('Western press vs Islamic Republic state media — multi-level comparison', fontsize=12)
    plt.tight_layout()
    out = COMPARE_DIR / 'compare_multilevel.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'\nSaved: {out.name}')

## Step 8: Validation checkpoint

In [ ]:
print('VALIDATION CHECKPOINT (compare_sources):')
print(f'  Western corpus  : {sum(en_arts.values())} articles, {len(en_edges)} edges')
print(f'  Iranian corpus  : {sum(ir_arts.values())} articles, {len(ir_edges)} edges')
print(f'  Iranian present : {IR_PRESENT}')
print()
outs = ['compare_corpus_stats.csv', 'compare_edges.csv',
        'compare_key_dyads.png', 'compare_actor_profiles.png',
        'compare_polarity.csv', 'compare_group_share.csv',
        'compare_graph_level.csv', 'compare_multilevel.png']
for f in outs:
    exists = (COMPARE_DIR / f).exists()
    print(f'    [{"OK" if exists else ("MISS" if IR_PRESENT else "skip")}]  {f}')
print()
print('Comparisons are per-article normalized or share-based (volume asymmetry')
print('handled). The SVO layer is NOT compared — the Iranian directed network rests')
print('on 40 actor->actor triples in total. The')
print('Iranian side is provisional until the alias audit (04_extract_iran misses)')
print('is done. Neither corpus was modified — outputs live only in data/output/compare/.')